In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================================
# 14. ICLE RESIDUAL-SIGNAL DIAGNOSTIC
# ============================================================================
# 11 and 12 found two real, documented ICLE corpus artifacts (literal
# self-disclosure, title-level topic exclusivity >= 80%). But after
# excluding BOTH from the LLM full-text/forced-JSON condition, the
# remaining ICLE subset (self-disclosure-clean, NOT topic-exclusive,
# n=71 on the real run) still shows ~98.6% accuracy -- statistically
# indistinguishable from the topic-exclusive group (~98.4%) and from raw
# ICLE accuracy (~98.75%). Neither audited confound explains why the LLM
# does so well on ICLE. This notebook looks for a THIRD candidate
# explanation, rather than assuming one, along four independent lines:
#
#   1. Continuous topic-share: the 80% exclusivity threshold is a hard
#      cutoff. Essays with e.g. 50-79% single-L1 title share are currently
#      lumped in with genuinely mixed-topic essays. If accuracy rises
#      smoothly with topic share below 80%, that's evidence of a graded
#      topic-correlation effect the binary threshold is hiding, not a
#      qualitatively different phenomenon.
#   2. Other ICLEv3 metadata fields (beyond Title) that might correlate
#      with L1 and leak through non-linguistic channels (e.g. institution,
#      exam type, age, sex) -- printed for inspection, not assumed.
#   3. Essay length: untimed (ICLE) vs timed (TOEFL11) essays may differ
#      systematically by L1 in ways a model could exploit without any
#      "real" NLI signal.
#   4. Distinctive terms within the n=71 residual group specifically, by
#      label -- concrete, inspectable leads for manual review, not proof
#      of a mechanism.
#
# This notebook does not conclude anything on its own -- it surfaces
# candidates. Read the printed output and decide what's worth chasing
# before rewriting the paper's Section 6 framing.

BASE_DIR = Path("..").resolve()
RESULTS_DIR = BASE_DIR / "data/processed/results"
COMBINED_CSV_PATH = BASE_DIR / "data/processed/combined_nli_masked.csv"
METADATA_PATH = BASE_DIR / "data/raw/ICLEV3/metadata.csv"
LEAKAGE_FLAGGED_PATH = RESULTS_DIR / "leakage_flagged_essays.csv"
FULLTEXT_LLM_PATH = RESULTS_DIR / "llm_predictions_fulltext.csv"

ICLE_LANG_MAP = {
    "German": "DEU", "French": "FRA", "Italian": "ITA", "Japanese": "JPN",
    "Korean": "KOR", "Spanish": "SPA", "Turkish": "TUR", "Chinese": "ZHO",
}
LANGS = ["DEU", "FRA", "ITA", "JPN", "KOR", "SPA", "TUR", "ZHO"]

if not METADATA_PATH.exists() or not FULLTEXT_LLM_PATH.exists():
    print(f"Skipping -- need both {METADATA_PATH.name} and {FULLTEXT_LLM_PATH.name}. Run 08 and check data/raw/ICLEV3 first.")
else:
    # Recomputed independently from raw metadata (not from 12's saved flag file), so this
    # notebook doesn't have to trust an intermediate -- same logic as 12, self-contained.
    meta = pd.read_csv(METADATA_PATH)
    print("ICLEv3 metadata columns:", list(meta.columns))

    meta_target = meta[meta["Native language"].isin(ICLE_LANG_MAP)].copy()
    meta_target["label"] = meta_target["Native language"].map(ICLE_LANG_MAP)
    meta_target["doc_id"] = "ICLEV3_" + meta_target["File name"].astype(str).str.strip()
    meta_target["Title"] = meta_target["Title"].fillna("MISSING_TITLE").astype(str).str.strip()

    title_stats = meta_target.groupby("Title")["label"].agg(
        n="size",
        top_label=lambda s: s.value_counts().idxmax(),
        top_count=lambda s: s.value_counts().max(),
    ).reset_index()
    title_stats["top_share"] = title_stats["top_count"] / title_stats["n"]
    meta_target = meta_target.merge(title_stats[["Title", "top_label", "top_share"]], on="Title", how="left")
    meta_target["title_matches_own_label"] = meta_target["top_label"] == meta_target["label"]
    # 0 when the title's dominant L1 isn't this doc's own label (no positive topic-correlation
    # signal for THIS doc's true label, regardless of how skewed the title is toward someone else).
    meta_target["own_label_topic_share"] = np.where(
        meta_target["title_matches_own_label"], meta_target["top_share"], 0.0
    )

    llm_df = pd.read_csv(FULLTEXT_LLM_PATH)
    llm_df = llm_df[llm_df["error"].fillna("").eq("") & llm_df["predicted_l1"].notna()].copy()
    if "eval_partition" in llm_df.columns:
        llm_df = llm_df[llm_df["eval_partition"] == "iclev3_external_test"].copy()
    else:
        print("WARNING: no eval_partition column -- cannot restrict to ICLE, results below may include TOEFL11.")

    self_ref_ids = set()
    if LEAKAGE_FLAGGED_PATH.exists():
        flagged = pd.read_csv(LEAKAGE_FLAGGED_PATH)
        self_ref_ids = set(flagged.loc[flagged["self_referential_in_text"] == True, "doc_id"])
    else:
        print(f"WARNING: {LEAKAGE_FLAGGED_PATH.name} not found -- run 11 first. Proceeding without excluding self-disclosure.")
    llm_df["self_referential_in_text"] = llm_df["doc_id"].isin(self_ref_ids)
    clean_llm = llm_df[~llm_df["self_referential_in_text"]].copy()

    clean_llm = clean_llm.merge(
        meta_target[["doc_id", "own_label_topic_share", "top_share", "title_matches_own_label"]],
        on="doc_id", how="left",
    )
    clean_llm["correct"] = clean_llm["predicted_l1"] == clean_llm["true_l1"]
    print(f"\nClean ICLE subset for this diagnostic: {len(clean_llm)} rows.")

    # ------------------------------------------------------------------
    # Check 1: continuous topic-share vs accuracy (not just the binary >=80% cutoff)
    # ------------------------------------------------------------------
    print("\n=== Check 1: LLM accuracy by own-label topic share (continuous, not binary) ===")
    bins = [-0.01, 0.0, 0.3, 0.5, 0.8, 1.01]
    bin_labels = ["0 (title dominated by a DIFFERENT L1)", "0.01-0.30", "0.31-0.50", "0.51-0.79", ">=0.80 (current cutoff)"]
    clean_llm["topic_share_bin"] = pd.cut(clean_llm["own_label_topic_share"], bins=bins, labels=bin_labels)
    by_bin = clean_llm.groupby("topic_share_bin", observed=False)["correct"].agg(n="size", accuracy="mean").reset_index()
    by_bin.to_csv(RESULTS_DIR / "icle_topic_share_continuous_accuracy.csv", index=False)
    print(by_bin.to_string(index=False))
    print(f"Saved: {RESULTS_DIR / 'icle_topic_share_continuous_accuracy.csv'}")
    print(
        "If accuracy rises smoothly across these bins, topic correlation is graded and the 80% cutoff is "
        "hiding a real (if weaker) effect below threshold. If the low bins are just as high as the top bin, "
        "topic share isn't the explanation and the residual is still unexplained."
    )

    # ------------------------------------------------------------------
    # Check 2: other ICLEv3 metadata fields, printed for inspection
    # ------------------------------------------------------------------
    print("\n=== Check 2: other ICLEv3 metadata fields by label ===")
    candidate_cols = [c for c in meta.columns if c not in ("Native language", "File name", "Title")]
    if not candidate_cols:
        print("No additional metadata columns beyond Native language / File name / Title.")
    for col in candidate_cols:
        nunique = meta_target[col].nunique(dropna=True)
        if 0 < nunique <= 30:
            ct = pd.crosstab(meta_target["label"], meta_target[col], normalize="index").round(3)
            print(f"\n--- {col} ({nunique} distinct values), row-normalized by label ---")
            print(ct)
        else:
            print(f"\n--- {col}: {nunique} distinct values -- too high-cardinality to crosstab, inspect manually if relevant ---")

    # ------------------------------------------------------------------
    # Check 3: essay length vs label / correctness
    # ------------------------------------------------------------------
    print("\n=== Check 3: essay length (word count) by dataset/label, and vs. correctness on the clean ICLE subset ===")
    if COMBINED_CSV_PATH.exists():
        combined = pd.read_csv(COMBINED_CSV_PATH)
        combined["word_count"] = combined["text"].fillna("").astype(str).str.split().apply(len)
        combined["dataset"] = combined["dataset"].astype(str).str.upper()
        combined["label"] = combined["label"].astype(str).str.upper()

        length_summary = combined.groupby(["dataset", "label"])["word_count"].median().unstack("label").reindex(columns=LANGS)
        length_summary.to_csv(RESULTS_DIR / "essay_word_count_median_by_dataset_label.csv")
        print("Median word count by dataset x label:")
        print(length_summary)
        print(f"Saved: {RESULTS_DIR / 'essay_word_count_median_by_dataset_label.csv'}")

        clean_llm = clean_llm.merge(combined[["doc_id", "word_count"]], on="doc_id", how="left")
        clean_llm["length_bin"] = pd.qcut(clean_llm["word_count"], q=4, duplicates="drop")
        by_length = clean_llm.groupby("length_bin", observed=False)["correct"].agg(n="size", accuracy="mean").reset_index()
        print("\nLLM accuracy by essay-length quartile, clean ICLE subset:")
        print(by_length.to_string(index=False))
    else:
        print(f"Skipping -- {COMBINED_CSV_PATH.name} not found.")

    # ------------------------------------------------------------------
    # Check 4: distinctive unigrams within the residual (non-topic-exclusive) group, by label
    # ------------------------------------------------------------------
    print("\n=== Check 4: distinctive terms in the residual (topic-share < 0.80) group, by label ===")
    residual = clean_llm[clean_llm["own_label_topic_share"] < 0.80].copy()
    if COMBINED_CSV_PATH.exists() and len(residual) >= 8:
        residual = residual.merge(combined[["doc_id", "text"]], on="doc_id", how="left")
        residual["text"] = residual["text"].fillna("").astype(str).str.lower()

        import re
        from collections import Counter

        def tokenize(s):
            return re.findall(r"[a-z]{3,}", s)

        per_label_counts = {}
        per_label_essay_n = {}
        overall_counts = Counter()
        for label, group in residual.groupby("true_l1"):
            c = Counter()
            for t in group["text"]:
                c.update(tokenize(t))
            per_label_counts[label] = c
            per_label_essay_n[label] = len(group)
            overall_counts.update(c)

        print(f"(computed over {len(residual)} residual-group essays -- small n, treat as leads, not conclusions)")
        for label, c in per_label_counts.items():
            others_total = sum(overall_counts.values()) - sum(c.values())
            scored = []
            for word, count in c.items():
                if count < 2:
                    continue
                other_count = overall_counts[word] - count
                ratio = (count / max(sum(c.values()), 1)) / ((other_count + 1) / max(others_total, 1))
                scored.append((word, count, round(ratio, 1)))
            scored.sort(key=lambda x: -x[2])
            top = scored[:10]
            print(f"\n{label} (n={per_label_essay_n[label]} essays): "
                  + ", ".join(f"{w}({cnt}x,{r}x more common)" for w, cnt, r in top))
    else:
        print("Skipping -- need combined corpus text and at least 8 residual-group rows.")

print("\nDone. This notebook is diagnostic only -- decide what to chase based on the printed output.")
